# 시드 반복 실험 — A(Weighted CE)·B(Focal Loss)

기존 A·B(시드 42)와 **데이터는 똑같이** 두고(문장 단위 9:1 분할, 10만 쌍 샘플 모두 시드 42 고정), **학습 시드만** 13, 2024로 바꿔 2번씩 더 학습합니다.
학습 시드가 바꾸는 것: 분류층 초기값, 배치 순서, 드롭아웃.

평가는 발표자료와 같은 **검증셋 전체 65,499쌍**. 마지막 셀에서 시드 42 결과와 합쳐 3회 평균 ± 표준편차를 만듭니다.

실행: 1 → 2 → 3 → 4 → 5 (오래 걸림, 중간에 끊겨도 다시 실행하면 끝난 것은 건너뜀) → 6

In [ ]:
# 1. 설정
import os, glob, json, random, gc
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import numpy as np
import pandas as pd
import torch

DATA_DIR    = "/home/jovyan/hallym_project2/Sample/02.라벨링데이터"   # v3 노트북과 같은 경로
MODEL_NAME  = "klue/roberta-base"
MAX_LEN     = 512
BATCH_SIZE  = 16
EPOCHS      = 3
LR          = 2e-5
DATA_SEED   = 42            # 분할·10만 샘플용 → 기존 A·B와 같은 데이터
MAX_SAMPLES = 100000
NEW_SEEDS   = [13, 2024]    # 새로 돌릴 학습 시드
FOCAL_GAMMA = 2.0
OUT_DIR     = "./seed_runs"
KEEP_MODEL  = False         # True면 각 런의 best 모델을 저장 (1개당 약 0.5GB)
os.makedirs(OUT_DIR, exist_ok=True)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "|", torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU")
json_files = sorted(glob.glob(os.path.join(DATA_DIR, "**", "*.json"), recursive=True))
print("JSON 파일 수:", len(json_files))

In [ ]:
# 2. 데이터 파싱 — v3 노트북 3번 셀과 같은 로직
TEXT_KEY, ASPECTS_KEY, ASPECT_KEY, POLARITY_KEY = "RawText", "Aspects", "Aspect", "SentimentPolarity"
POL_MAP = {-1: "부정", 0: "중립", 1: "긍정"}

def norm_polarity(v):
    try:
        return POL_MAP.get(int(float(v)), str(v))
    except (TypeError, ValueError):
        return str(v).strip()

def iter_records(obj):
    if isinstance(obj, list):
        for x in obj: yield from iter_records(x)
    elif isinstance(obj, dict):
        if TEXT_KEY in obj:
            yield obj
        else:
            for v in obj.values():
                if isinstance(v, (list, dict)): yield from iter_records(v)

def split_of(path):
    p = path.lower()
    if "validation" in p or "valid" in p or "검증" in p: return "valid"
    return "train"

rows, bad = [], 0
for path in json_files:
    try:
        with open(path, encoding="utf-8-sig") as f:
            obj = json.load(f)
    except Exception:
        bad += 1; continue
    for rec in iter_records(obj):
        text = str(rec.get(TEXT_KEY, "")).strip()
        for a in rec.get(ASPECTS_KEY) or []:
            aspect = str(a.get(ASPECT_KEY, "")).strip()
            if text and aspect and a.get(POLARITY_KEY) is not None:
                rows.append({"text": text, "aspect": aspect,
                             "label": norm_polarity(a[POLARITY_KEY]), "split": split_of(path)})

df = pd.DataFrame(rows)
keys = ["text", "aspect", "split"]
cnt = df.groupby(keys + ["label"]).size().reset_index(name="n")
cnt = cnt.sort_values(keys + ["n"], ascending=[True, True, True, False])
cnt["rank"] = cnt.groupby(keys).cumcount()
first  = cnt[cnt["rank"] == 0]
second = cnt[cnt["rank"] == 1][keys + ["n"]].rename(columns={"n": "n2"})
m = first.merge(second, on=keys, how="left")
df = m[m["n2"].isna() | (m["n"] > m["n2"])][keys + ["label"]].reset_index(drop=True)
del rows, cnt, first, second, m; gc.collect()
print(f"읽기 실패 {bad} | 정리 후 {len(df):,}쌍  (기대값 655,440)")

In [ ]:
# 3. 분할 — 기존 A·B와 완전히 같은 데이터 (시드 42 고정)
if (df["split"] == "valid").any() and (df["split"] == "train").any():
    train_full = df[df["split"] == "train"].reset_index(drop=True)
    valid_full = df[df["split"] == "valid"].reset_index(drop=True)
else:
    texts = np.array(df["text"].unique(), dtype=object)
    rng = np.random.default_rng(DATA_SEED)
    rng.shuffle(texts)
    valid_texts = set(texts[: int(len(texts) * 0.1)])
    is_valid = df["text"].isin(valid_texts)
    train_full, valid_full = df[~is_valid].reset_index(drop=True), df[is_valid].reset_index(drop=True)

train_df  = train_full.sample(min(MAX_SAMPLES, len(train_full)), random_state=DATA_SEED).reset_index(drop=True)
valid_sel = valid_full.sample(min(MAX_SAMPLES // 5, len(valid_full)), random_state=DATA_SEED).reset_index(drop=True)  # 에폭별 best 선택용 (기존과 동일)

LABELS = sorted(df["label"].unique())
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}
del train_full, df; gc.collect()
print("labels:", label2id)
print(f"train {len(train_df):,} | 선택용 valid {len(valid_sel):,} | 최종 평가 valid {len(valid_full):,} (기대값 65,499)")

In [ ]:
# 4. 데이터셋 · 손실함수 · 학습/평가 함수
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, get_linear_schedule_with_warmup)
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support
from tqdm.auto import tqdm

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
collator  = DataCollatorWithPadding(tokenizer)

class ABSADataset(Dataset):
    def __init__(self, frame):
        self.aspects = frame["aspect"].tolist(); self.texts = frame["text"].tolist()
        self.labels  = [label2id[l] for l in frame["label"]]
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        item = dict(tokenizer(self.aspects[i], self.texts[i], truncation="only_second",
                              max_length=MAX_LEN, return_token_type_ids=False))
        item["labels"] = self.labels[i]
        return item

train_ds, sel_ds, full_ds = ABSADataset(train_df), ABSADataset(valid_sel), ABSADataset(valid_full)
counts = train_df["label"].map(label2id).value_counts().sort_index().values
class_w = torch.tensor(len(train_df) / (len(LABELS) * counts), dtype=torch.float).to(device)
print("class weights:", dict(zip(LABELS, class_w.cpu().numpy().round(3))))

class FocalLoss(torch.nn.Module):
    """FL = -α_y (1 - p_y)^γ log p_y,  α = Weighted CE와 같은 클래스 가중치"""
    def __init__(self, alpha, gamma=2.0):
        super().__init__(); self.alpha, self.gamma = alpha, gamma
    def forward(self, logits, y):
        logp = F.log_softmax(logits, -1).gather(1, y[:, None]).squeeze(1)
        return (-self.alpha[y] * (1 - logp.exp()) ** self.gamma * logp).mean()

use_amp   = device.type == "cuda"
amp_dtype = torch.bfloat16 if use_amp and torch.cuda.is_bf16_supported() else torch.float16

@torch.no_grad()
def predict(model, ds):
    model.eval(); preds = []
    dl = DataLoader(ds, batch_size=BATCH_SIZE * 4, shuffle=False, collate_fn=collator, num_workers=2)
    for batch in tqdm(dl, desc="eval", leave=False):
        batch = {k: v.to(device) for k, v in batch.items()}; batch.pop("labels")
        with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
            preds += model(**batch).logits.argmax(-1).tolist()
    return [id2label[p] for p in preds]

def scores(gold, pred):
    L = ["긍정", "부정", "중립"]
    P, R, Fs, _ = precision_recall_fscore_support(gold, pred, labels=L, zero_division=0)
    return {"macro_f1": f1_score(gold, pred, labels=L, average="macro"), "accuracy": accuracy_score(gold, pred),
            "f1_긍정": Fs[0], "f1_부정": Fs[1], "f1_중립": Fs[2], "P_중립": P[2], "R_중립": R[2]}

def run(model_key, seed):
    set_seed(seed)                                   # 학습 시드: 초기값·배치 순서·드롭아웃
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=len(LABELS), id2label=id2label, label2id=label2id).to(device)
    loss_fn = torch.nn.CrossEntropyLoss(weight=class_w) if model_key == "A" else FocalLoss(class_w, FOCAL_GAMMA)
    g = torch.Generator(); g.manual_seed(seed)
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, generator=g,
                          collate_fn=collator, num_workers=2)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    total = len(train_dl) * EPOCHS
    sch = get_linear_schedule_with_warmup(opt, int(total * 0.1), total)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp and amp_dtype == torch.float16)

    best_f1, best_state, hist = -1, None, []
    for ep in range(1, EPOCHS + 1):
        model.train(); running = 0.0
        for step, batch in enumerate(tqdm(train_dl, desc=f"{model_key} seed{seed} ep{ep}"), 1):
            batch = {k: v.to(device) for k, v in batch.items()}; y = batch.pop("labels")
            with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
                loss = loss_fn(model(**batch).logits.float(), y)
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sch.step(); running += loss.item()
        f1 = f1_score(valid_sel["label"], predict(model, sel_ds), average="macro")
        hist.append(round(f1, 4)); print(f"  ep{ep} 선택용 macro-F1 {f1:.4f}")
        if f1 > best_f1:                              # 기존과 같이 에폭별 best 선택
            best_f1 = f1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    if KEEP_MODEL:
        model.save_pretrained(f"{OUT_DIR}/{model_key}_s{seed}"); tokenizer.save_pretrained(f"{OUT_DIR}/{model_key}_s{seed}")
    res = {"model": model_key, "seed": seed, "epoch_f1": str(hist),
           **scores(valid_full["label"], predict(model, full_ds))}
    del model, opt, best_state; gc.collect(); torch.cuda.empty_cache()
    return res

In [ ]:
# 5. 학습 실행 (4회: A×2, B×2). 끝난 런은 seed_results.csv에 저장되어 다시 실행해도 건너뜀
res_path = f"{OUT_DIR}/seed_results.csv"
done = pd.read_csv(res_path) if os.path.exists(res_path) else pd.DataFrame(columns=["model", "seed"])

for model_key in ["A", "B"]:
    for seed in NEW_SEEDS:
        if ((done["model"] == model_key) & (done["seed"] == seed)).any():
            print(f"{model_key} seed {seed}: 이미 완료 → 건너뜀"); continue
        r = run(model_key, seed)
        print(f"▶ {model_key} seed {seed}: macro-F1 {r['macro_f1']:.4f} | 중립 F1 {r['f1_중립']:.4f}")
        done = pd.concat([done, pd.DataFrame([r])], ignore_index=True)
        done.to_csv(res_path, index=False)
done

In [ ]:
# 6. 시드 42(기존 재평가 값)와 합쳐 3회 평균 ± 표준편차
base42 = pd.DataFrame([   # eval_results/scores_full_valid.csv 값 (검증셋 65,499)
    {"model": "A", "seed": 42, "macro_f1": 0.8335, "accuracy": 0.9513, "f1_긍정": 0.9778, "f1_부정": 0.9275,
     "f1_중립": 0.5953, "P_중립": 0.6152, "R_중립": 0.5767},
    {"model": "B", "seed": 42, "macro_f1": 0.8352, "accuracy": 0.9505, "f1_긍정": 0.9771, "f1_부정": 0.9292,
     "f1_중립": 0.5994, "P_중립": 0.5819, "R_중립": 0.6180},
])
allr = pd.concat([base42, pd.read_csv(res_path)], ignore_index=True)
cols = ["macro_f1", "accuracy", "f1_중립", "P_중립", "R_중립"]
print("■ 시드별"); display(allr[["model", "seed"] + cols].sort_values(["model", "seed"]).round(4))

summ = allr.groupby("model")[cols].agg(["mean", "std"])
print("■ 평균 ± 표준편차 (n=%s)" % allr.groupby("model").size().to_dict())
for mk in summ.index:
    print(mk, " | ".join(f"{c} {summ.loc[mk,(c,'mean')]:.4f}±{summ.loc[mk,(c,'std')]:.4f}" for c in cols))

piv = allr.pivot(index="seed", columns="model", values="macro_f1")
diff = (piv["B"] - piv["A"]) * 100
print("\n■ 같은 시드끼리 B − A (macro-F1, %p):", diff.round(2).to_dict())
print(f"  평균 {diff.mean():+.2f}%p ± {diff.std():.2f}  → 0을 사이에 두면 'Focal 효과 없음'이 확정")

summ.round(4).to_csv(f"{OUT_DIR}/seed_summary.csv")
allr.round(4).to_csv(f"{OUT_DIR}/seed_all.csv", index=False)
print("저장:", f"{OUT_DIR}/seed_summary.csv", f"{OUT_DIR}/seed_all.csv")